**Customers**

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

df_customers = spark.table("medallion.bronze.tb_customers")

df_customers = df_customers.select(
    F.col("customer_id").alias("id_consumidor"),
    F.col("customer_unique_id").alias("id_consumidor_unico"),
    F.col("customer_zip_code_prefix").alias("prefixo_cep"),
    F.col("customer_city").alias("cidade"),
    F.col("customer_state").alias("estado"),
    F.col("customer_name").alias("nome_consumidor"),
    F.col("customer_gender").alias("genero_consumidor"),
    F.col("customer_birth_date").alias("data_aniversario_consumidor"),
    F.col("customer_age").alias("idade_consumidor"),
    "timestamp_ingestion"
)

df_customers = df_customers.withColumn("cidade", F.upper("cidade")) \
       .withColumn("estado", F.upper("estado"))

df_customers = df_customers.filter(F.col("id_consumidor").isNotNull())

window_spec = Window.partitionBy("id_consumidor") \
    .orderBy(
        F.col("timestamp_ingestion").desc()
    )

df_customers = df_customers.withColumn(
    "row_number",
    F.row_number().over(window_spec)
)

df_customers = df_customers.filter(F.col("row_number") == 1).drop("row_number")

spark.sql(f"create schema if not exists silver")
df_customers.write \
  .format("delta") \
  .mode("overwrite") \
  .option("overwriteSchema", "true") \
  .saveAsTable("medallion.silver.dim_consumidores")

**Orders**

In [0]:
df_orders = spark.table("medallion.bronze.tb_orders")

df_orders = df_orders.select(
    F.col("order_id").alias("id_pedido"),
    F.col("customer_id").alias("id_consumidor"),
    F.col("order_status").alias("status_pedido"),
    F.col("order_purchase_timestamp").alias("data_compra"),
    F.col("order_approved_at").alias("data_aprovacao"),
    F.col("order_delivered_carrier_date").alias("data_envio_transportadora"),
    F.col("order_delivered_customer_date").alias("data_entrega_cliente"),
    F.col("order_estimated_delivery_date").alias("data_entrega_estimada"),
    "timestamp_ingestion"
)

df_orders = df_orders.withColumn(
    "status_pedido",
    F.when(F.col("status_pedido") == "delivered", "entregue")
     .when(F.col("status_pedido") == "canceled", "cancelado")
     .when(F.col("status_pedido") == "shipped", "enviado")
     .when(F.col("status_pedido") == "processing", "processando")
     .when(F.col("status_pedido") == "invoiced", "faturado")
     .when(F.col("status_pedido") == "unavailable", "indisponível")
     .when(F.col("status_pedido") == "created", "criado")
     .when(F.col("status_pedido") == "approved", "aprovado")
     .otherwise("desconhecido")
)

df_orders = df_orders \
.withColumn("data_compra", F.to_timestamp("data_compra")) \
.withColumn("data_aprovacao", F.to_timestamp("data_aprovacao")) \
.withColumn("data_envio_transportadora", F.to_timestamp("data_envio_transportadora")) \
.withColumn("data_entrega_cliente", F.to_timestamp("data_entrega_cliente")) \
.withColumn("data_entrega_estimada", F.to_timestamp("data_entrega_estimada"))

df_orders = df_orders.withColumn(
    "tempo_entrega_dias",
    F.datediff(
        F.col("data_entrega_cliente"),
        F.col("data_compra")
    )
)

df_orders = df_orders.withColumn(
    "tempo_entrega_dias",
    F.when(F.col("tempo_entrega_dias") < 0, None)
     .otherwise(F.col("tempo_entrega_dias"))
)

df_orders = df_orders.withColumn(
    "tempo_entrega_estimado_dias",
    F.datediff(
        F.col("data_entrega_estimada"),
        F.col("data_compra")
    )
)

df_orders = df_orders.withColumn(
    "diferenca_entrega_dias",
    F.col("tempo_entrega_dias") - F.col("tempo_entrega_estimado_dias")
)

df_orders = df_orders.withColumn(
    "entrega_no_prazo",
    F.when(F.col("status_pedido") != "entregue", "Não Entregue")
     .when(F.col("tempo_entrega_dias") <= F.col("tempo_entrega_estimado_dias"), "Sim")
     .otherwise("Não")
)

spark.sql(f"create schema if not exists silver")
df_orders.write \
  .format("delta") \
  .mode("overwrite") \
  .option("overwriteSchema", "true") \
  .saveAsTable("medallion.silver.fat_pedidos")

Order Items

In [0]:
df_items = spark.table("medallion.bronze.tb_order_items")

df_items = df_items.select(
    F.col("order_id").alias("id_pedido"),
    F.col("order_item_id").alias("id_item"),
    F.col("product_id").alias("id_produto"),
    F.col("seller_id").alias("id_vendedor"),
    F.col("shipping_limit_date").alias("data_entrega_limite"),
    F.col("price").alias("preco_BRL"),
    F.col("freight_value").alias("preco_frete"),
    "timestamp_ingestion"
)

df_items = df_items.withColumn("preco_BRL", F.col("preco_BRL").cast("double")) \
                   .withColumn("preco_frete", F.col("preco_frete").cast("double"))

df_items.write \
  .format("delta") \
  .mode("overwrite") \
  .option("overwriteSchema", "true") \
  .saveAsTable("medallion.silver.fat_itens_pedidos")

Order Payments

In [0]:
df_payments = spark.table("medallion.bronze.tb_order_payments")

df_payments = df_payments.select(
    F.col("order_id").alias("id_pedido"),
    F.col("payment_sequential").alias("n_parcela"),
    F.col("payment_type").alias("tipo_pagamento"),
    F.col("payment_installments").alias("qtd_parcelas"),
    F.col("payment_value").alias("valor_pagamento"),
    "timestamp_ingestion"
)

df_payments = df_payments.withColumn(
    "tipo_pagamento",
    F.when(F.col("tipo_pagamento") == "credit_card", "Cartão de Crédito")
     .when(F.col("tipo_pagamento") == "boleto", "Boleto")
     .when(F.col("tipo_pagamento") == "voucher", "Voucher")
     .when(F.col("tipo_pagamento") == "debit_card", "Cartão de Débito")
     .when(F.col("tipo_pagamento") == "not_defined", "Não Definido")
     .otherwise("Outros")
)

df_payments = df_payments \
    .withColumn("qtd_parcelas", F.col("qtd_parcelas").cast("int")) \
    .withColumn("n_parcela", F.col("n_parcela").cast("int")) \
    .withColumn("valor_pagamento", F.col("valor_pagamento").cast("double"))

df_payments.write \
  .format("delta") \
  .mode("overwrite") \
  .option("overwriteSchema", "true") \
  .saveAsTable("medallion.silver.fat_pagamentos_pedidos")

**Order Review**

In [0]:
df_reviews = spark.table("medallion.bronze.tb_order_reviews")

df_reviews = df_reviews.select(
    F.col("review_id").cast("string").alias("id_avaliacao"),
    F.col("order_id").cast("string").alias("id_pedido"),
    F.expr("try_cast(review_score as int)").alias("nota_avaliacao"),
    F.col("review_comment_title").cast("string").alias("titulo_comentario_avaliacao"),
    F.col("review_comment_message").cast("string").alias("comentario_avaliacao"),
    F.col("review_creation_date").alias("data_criacao_avaliacao"),
    F.col("review_answer_timestamp").alias("data_resposta_avaliacao"),
    F.col("timestamp_ingestion").cast("timestamp")
)

df_reviews = df_reviews.withColumn(
    "data_criacao_avaliacao",
    F.expr("try_to_timestamp(data_criacao_avaliacao)")
).withColumn(
    "data_resposta_avaliacao",
    F.expr("try_to_timestamp(data_resposta_avaliacao)")
)

df_reviews = df_reviews.filter(F.col("id_pedido").isNotNull())

df_reviews = df_reviews.filter(
    (F.col("data_criacao_avaliacao").isNotNull()) &
    (F.col("data_criacao_avaliacao") <= F.current_timestamp())
)

df_reviews = df_reviews.fillna({
    "comentario_avaliacao": "Sem comentário",
    "titulo_comentario_avaliacao": "Sem título"
})

df_reviews = df_reviews.withColumn(
    "comentario_avaliacao",
    F.when(F.col("comentario_avaliacao") == "", "Sem comentário")
     .otherwise(F.col("comentario_avaliacao"))
).withColumn(
    "titulo_comentario_avaliacao",
    F.when(F.col("titulo_comentario_avaliacao") == "", "Sem título")
     .otherwise(F.col("titulo_comentario_avaliacao"))
)

df_reviews = df_reviews.withColumn(
    "timestamp_ingestion",
    F.coalesce(F.col("timestamp_ingestion"), F.current_timestamp())
)

spark.sql("create schema if not exists silver")

df_reviews.write \
  .format("delta") \
  .mode("overwrite") \
  .option("overwriteSchema", "true") \
  .saveAsTable("medallion.silver.fat_avaliacoes_pedidos")

**Products**

In [0]:
df_produtos = spark.table("medallion.bronze.tb_products")

df_produtos = df_produtos.select(
    F.col("product_id").alias("id_produto"),
    F.col("product_name").alias("nome_produto"),
    F.col("product_category_name").alias("categoria_produto"),
    F.col("product_name_lenght").alias("tamanho_nome_produto"),
    F.col("product_description_lenght").alias("tamanho_descricao_produto"),
    F.col("product_photos_qty").alias("quantidade_fotos"),
    F.col("product_weight_g").alias("peso_produto_gramas"),
    F.col("product_length_cm").alias("comprimento_centimetros"),
    F.col("product_height_cm").alias("altura_centimetros"),
    F.col("product_width_cm").alias("largura_centimetros"),
    "timestamp_ingestion"
)

df_produtos = df_produtos \
    .withColumn("tamanho_nome_produto", F.col("tamanho_nome_produto").cast("int")) \
    .withColumn("tamanho_descricao_produto", F.col("tamanho_descricao_produto").cast("int")) \
    .withColumn("quantidade_fotos", F.col("quantidade_fotos").cast("int")) \
    .withColumn("peso_produto_gramas", F.col("peso_produto_gramas").cast("int")) \
    .withColumn("comprimento_centimetros", F.col("comprimento_centimetros").cast("int")) \
    .withColumn("altura_centimetros", F.col("altura_centimetros").cast("int")) \
    .withColumn("largura_centimetros", F.col("largura_centimetros").cast("int"))

window_spec = Window.partitionBy("id_produto") \
    .orderBy(F.col("timestamp_ingestion").desc())

df_produtos = df_produtos.withColumn(
    "row_number",
    F.row_number().over(window_spec)
).filter("row_number = 1").drop("row_number")

df_produtos = df_produtos.withColumn(
    "categoria_produto",
    F.when(F.col("categoria_produto").isNull(), "Sem Categoria")
     .otherwise(F.col("categoria_produto"))
)

spark.sql("create schema if not exists silver")
df_produtos.write \
  .format("delta") \
  .mode("overwrite") \
  .option("overwriteSchema", "true") \
  .saveAsTable("medallion.silver.dim_produtos")

**Sellers**

In [0]:
df_sellers = spark.table("medallion.bronze.tb_sellers")

df_sellers = df_sellers.select(
    F.col("seller_id").alias("id_vendedor"),
    F.col("seller_zip_code_prefix").alias("prefixo_cep"),
    F.col("seller_city").alias("cidade"),
    F.col("seller_state").alias("estado"),
    F.col("seller_name").alias("nome_vendedor"),
    F.col("seller_registration_date").alias("data_registro_vendedor"),
    "timestamp_ingestion"
)

df_sellers = df_sellers.withColumn("cidade", F.upper("cidade")) \
                       .withColumn("estado", F.upper("estado"))

df_sellers = df_sellers.filter(F.col("id_vendedor").isNotNull())

window_spec = Window.partitionBy("id_vendedor") \
    .orderBy(F.col("timestamp_ingestion").desc())

df_sellers = df_sellers.withColumn(
    "row_number",
    F.row_number().over(window_spec)
).filter("row_number = 1").drop("row_number")

window_spec = Window.partitionBy("id_vendedor") \
    .orderBy(F.col("timestamp_ingestion").desc())

df_sellers = df_sellers.withColumn(
    "row_number",
    F.row_number().over(window_spec)
).filter("row_number = 1").drop("row_number")

spark.sql("create schema if not exists silver")
df_sellers.write \
  .format("delta") \
  .mode("overwrite") \
  .option("overwriteSchema", "true") \
  .saveAsTable("medallion.silver.dim_vendedores")

Product Category Name

In [0]:
df_pcn = spark.table("medallion.bronze.tb_product_category_name_translation")

df_pcn = df_pcn.select(
    F.col("product_category_name").alias("nome_produto_pt"),
    F.col("product_category_name_english").alias("nome_produto_en"),
    "timestamp_ingestion"
)

df_pcn = df_pcn.dropDuplicates()

spark.sql("create schema if not exists silver")
df_pcn.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("medallion.silver.dim_categoria_produtos_traducao")

Cotação Dólar

In [0]:
df_dolar = spark.table("medallion.bronze.tb_cotacao_dolar")
df_dolar = df_dolar.withColumn("data", F.to_date("dataHoraCotacao"))

datas = df_dolar.agg(
    F.min("data").alias("min_data"),
    F.max("data").alias("max_data")
).collect()[0]

min_data = datas["min_data"]
max_data = datas["max_data"]

df_calendario = spark.createDataFrame(
    [(min_data, max_data)],
    ["start", "end"]
).select(
    F.explode(F.sequence("start", "end")).alias("data")
)

df_join = df_calendario.join(
    df_dolar.select("data", "cotacaoCompra"),
    on="data",
    how="left"
)

window_spec = Window.orderBy("data") \
    .rowsBetween(Window.unboundedPreceding, 0)

df_final = df_join.withColumn(
    "cotacao_preenchida",
    F.last("cotacaoCompra", ignorenulls=True).over(window_spec)
)

df_final = df_final.select(
    F.col("data"),
    F.col("cotacao_preenchida").alias("cotacao_dolar"),
    F.current_timestamp().alias("timestamp_ingestion")
)

spark.sql("create schema if not exists silver")
df_final.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("medallion.silver.dim_cotacao_dolar")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Total Pedido

In [0]:
%sql
use catalog medallion;

In [0]:
%sql
create or replace table silver.fat_pedido_total as

with pagamentos as (
    select 
        id_pedido,
        sum(valor_pagamento) as valor_total_pago_brl
    from silver.fat_pagamentos_pedidos
    group by id_pedido
)

select
    p.id_pedido,
    p.id_consumidor,
    p.status_pedido as status,
    
    cast(round(pg.valor_total_pago_brl, 2) as decimal(10,2)) as valor_total_pago_brl,
    
    cast(
        round(pg.valor_total_pago_brl / nullif(c.cotacao_dolar, 0), 2)
        as decimal(10,2)
    ) as valor_total_pago_usd,
    
    p.data_compra as data_pedido

from silver.fat_pedidos p

left join pagamentos pg
    on p.id_pedido = pg.id_pedido

left join silver.dim_cotacao_dolar c
    on to_date(p.data_compra) = c.data;

num_affected_rows,num_inserted_rows
